
# Comandos Mágicos

Os comandos mágicos são comandos executáveis em notebooks Databricks para simplificar a tarefas de configuração de ambiente, interagir com banco de dados e gerenciar dados como um todo. São iniciados pelo prefixo % e executados em células Python (geralmente)


## 1. %md - Markdown

O comando `%md` permite criar células de documentação usando **Markdown**.

- Suporta formatação de texto
- Listas e tabelas
- Links e imagens
- Código inline com `backticks`

**Exemplo de tabela:**

| Comando | Descrição |
|---------|----------|
| %python | Executa código Python |
| %sql    | Executa queries SQL |
| %sh     | Executa comandos shell |

In [0]:
# 2. %python - Código Python
# Este é o comando padrão em notebooks Python
# Permite executar qualquer código Python

import pandas as pd
from datetime import datetime

# Criar dados de exemplo
data = {
    'produto': ['Laptop', 'Mouse', 'Teclado', 'Monitor'],
    'preco': [3500.00, 85.50, 245.00, 1200.00],
    'quantidade': [10, 50, 30, 15]
}

df_pandas = pd.DataFrame(data)
print("DataFrame Pandas:")
print(df_pandas)
print(f"\nData de execução: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# Converter para Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)

# Mostrar o DataFrame
print("Spark DataFrame criado com sucesso!")
display(df_spark)

In [0]:
%sql
-- 3. %sql - Executar SQL
-- Permite executar queries SQL diretamente
-- Cria uma view temporária do DataFrame anterior

CREATE OR REPLACE TEMP VIEW produtos_temp AS
SELECT 
  'Laptop' as produto,
  3500.00 as preco,
  10 as quantidade
UNION ALL
SELECT 'Mouse', 85.50, 50
UNION ALL
SELECT 'Teclado', 245.00, 30
UNION ALL
SELECT 'Monitor', 1200.00, 15;

-- Consultar os dados
SELECT 
  produto,
  preco,
  quantidade,
  preco * quantidade as valor_total
FROM produtos_temp
ORDER BY valor_total DESC;

In [0]:
%sql
-- Agregações em SQL
SELECT 
  COUNT(*) as total_produtos,
  SUM(quantidade) as quantidade_total,
  AVG(preco) as preco_medio,
  MAX(preco) as preco_maximo,
  MIN(preco) as preco_minimo,
  SUM(preco * quantidade) as receita_total
FROM produtos_temp;

In [0]:
%sh
# 4. %sh - Comandos Shell
# Permite executar comandos do sistema operacional

echo "=== Informações do Sistema ==="
echo ""
echo "Diretório atual:"
pwd
echo ""
echo "Usuário:"
whoami
echo ""
echo "Data e hora:"
date
echo ""
echo "Espaço em disco:"
df -h | head -5

In [0]:
# 5. %fs - Comandos de Sistema de Arquivos
# Permite interagir com DBFS (Databricks File System)

# Listar diretórios (usando dbutils ao invés de %fs magic)
print("=== Comandos de Sistema de Arquivos ===")
print("\nDiretórios disponíveis em /databricks-datasets:")

try:
    files = dbutils.fs.ls("/databricks-datasets")
    for f in files[:5]:  # Mostrar apenas os primeiros 5
        print(f"{f.path} - {f.name}")
except Exception as e:
    print(f"Aviso: {e}")
    print("\nExemplo de comandos %fs disponíveis:")
    print("  %fs ls /path/to/directory")
    print("  %fs head /path/to/file")
    print("  %fs mkdirs /path/to/new/directory")
    print("  %fs rm /path/to/file")

In [0]:
%pip install faker

# O comando %pip permite instalar pacotes Python diretamente no notebook
# Diferente de !pip, o %pip instala no ambiente correto do cluster

In [0]:
# Usar o pacote instalado
from faker import Faker

fake = Faker('pt_BR')

print("=== Dados Faker Gerados ===")
print(f"Nome: {fake.name()}")
print(f"Endereço: {fake.address()}")
print(f"Email: {fake.email()}")
print(f"Telefone: {fake.phone_number()}")
print(f"Empresa: {fake.company()}")

# Gerar dados em massa
print("\n=== Lista de 5 pessoas ===")
for i in range(5):
    print(f"{i+1}. {fake.name()} - {fake.email()}")

In [0]:
# 6. Combinando SQL e Python
# Você pode executar SQL inline com spark.sql()

resultado_sql = spark.sql("""
    SELECT 
        produto,
        preco,
        quantidade,
        preco * quantidade as valor_total
    FROM produtos_temp
    WHERE preco > 200
""")

print("Produtos com preço acima de R$ 200:")
display(resultado_sql)

# Converter para Pandas para análise
df_resultado = resultado_sql.toPandas()
print(f"\nTotal de produtos filtrados: {len(df_resultado)}")
print(f"Valor médio: R$ {df_resultado['preco'].mean():.2f}")

In [0]:
# 7. Widgets - Parâmetros de Notebook
# Criar widgets para entrada de dados dinâmica

dbutils.widgets.text("nome_produto", "Laptop", "Nome do Produto")
dbutils.widgets.dropdown("categoria", "Eletrônicos", ["Eletrônicos", "Móveis", "Escritório", "Acessórios"], "Categoria")

# Obter valores dos widgets
produto = dbutils.widgets.get("nome_produto")
categoria = dbutils.widgets.get("categoria")

print(f"Produto selecionado: {produto}")
print(f"Categoria: {categoria}")

# Os widgets aparecem no topo do notebook e permitem entrada dinâmica

## Resumo dos Comandos Mágicos

### Comandos de Linguagem
| Comando | Descrição | Disponível |
|---------|-----------|------------|
| `%python` | Executa código Python | ✅ |
| `%sql` | Executa queries SQL | ✅ |
| `%sh` | Executa comandos shell | ✅ |
| `%md` | Renderiza Markdown | ✅ |
| `%r` | Executa código R | ❌ (não no serverless) |
| `%scala` | Executa código Scala | ❌ (não no serverless) |

### Comandos Utilitários
| Comando | Descrição |
|---------|----------|
| `%fs` | Operações de sistema de arquivos DBFS |
| `%pip` | Gerenciamento de pacotes Python |
| `%run` | Executa outro notebook |

### Dicas Importantes

1. **%pip vs !pip**: Use `%pip` ao invés de `!pip` para garantir instalação no ambiente correto
2. **Células SQL**: Resultados de SQL são automaticamente salvos em `_sqldf`
3. **dbutils**: Biblioteca utilitária do Databricks para widgets, filesystem, secrets, etc.
4. **Mixing languages**: Você pode misturar diferentes linguagens no mesmo notebook
5. **display()**: Use `display()` para visualizações interativas de DataFrames

In [0]:
# 8. Exemplo Prático - Pipeline ETL Simples
print("=== ETL Pipeline Example ===")

# Extract: Criar dados de exemplo
from datetime import datetime, timedelta
import random

data_vendas = []
for i in range(100):
    data_vendas.append({
        'id': i + 1,
        'data': (datetime.now() - timedelta(days=random.randint(0, 30))).strftime('%Y-%m-%d'),
        'produto': random.choice(['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Webcam']),
        'quantidade': random.randint(1, 10),
        'preco_unitario': round(random.uniform(50, 3500), 2)
    })

# Transform: Criar DataFrame Spark
df_vendas = spark.createDataFrame(data_vendas)

print("✓ Extract: Dados gerados")
print("✓ Transform: DataFrame criado")
print(f"Total de registros: {df_vendas.count()}")

In [0]:
# Load: Criar view temporária para análise SQL
df_vendas.createOrReplaceTempView("vendas")

print("✓ Load: View temporária 'vendas' criada")
print("\nPrimeiras 10 vendas:")
display(df_vendas.limit(10))

In [0]:
%sql
-- Análise: Vendas por produto
SELECT 
    produto,
    COUNT(*) as num_vendas,
    SUM(quantidade) as quantidade_total,
    ROUND(AVG(preco_unitario), 2) as preco_medio,
    ROUND(SUM(quantidade * preco_unitario), 2) as receita_total
FROM vendas
GROUP BY produto
ORDER BY receita_total DESC;

## Conclusão

Os **comandos mágicos** do Databricks são ferramentas poderosas que:

✅ **Simplificam** o desenvolvimento de notebooks

✅ **Permitem** misturar múltiplas linguagens

✅ **Facilitam** operações de sistema de arquivos e gerenciamento de pacotes

✅ **Aumentam** a produtividade com widgets e parâmetros

### Próximos Passos

- Explore o DBFS com `%fs` ou `dbutils.fs`
- Crie notebooks reutilizáveis com `%run`
- Configure widgets para notebooks parametrizados
- Combine Python, SQL e Shell para pipelines completos

---

**Documentação Oficial**: [Databricks Magic Commands](https://docs.databricks.com/notebooks/notebooks-use.html#mix-languages)

In [0]:
%fs

In [0]:
%fs ls

In [0]:
%fs ls databricks-datasets

In [0]:
%lsmagic